# Deadline Predictor V4.7 - Ensemble & Log-Transform

## 🎯 Goal
Push R² > 0.6 using **Ensemble Learning (Voting)** and **Target Log-Transformation**.

## 🚀 Updates (V4.7)
1.  **Log-Transformation**: Time data (days) differs by orders of magnitude. Training on `log(days)` often helps.
2.  **Voting Regressor**: Combines the stability of **Linear Regression** with the power of **Random Forest**.
3.  **XGBoost Constraint**: Simplified hyperparameters to prevent the previous overfitting (R²=0.02).

In [34]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

In [35]:
# -----------------------------
# 1️⃣ CONFIG
# -----------------------------
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)
SPRINTS_FILE = r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Sprints 41.csv"
ISSUES_FILE = r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues 554.csv"
SUMMARY_FILE = r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues summery 568.csv"
PRIORITY_MAP = {"Low": 1, "Medium": 2, "High": 3, "Showstopper": 5}

In [36]:
# -----------------------------
# 2️⃣ DATA LOADING
# -----------------------------
def load_and_prep_enhanced():
    spr = pd.read_csv(SPRINTS_FILE)
    iss = pd.read_csv(ISSUES_FILE, on_bad_lines="skip")
    summ = pd.read_csv(SUMMARY_FILE, on_bad_lines="skip")
    for df in [spr, iss, summ]: df.columns = [c.lower().strip() for c in df.columns]
    
    spr["sprintstartdate"] = pd.to_datetime(spr["sprintstartdate"], errors="coerce")
    spr["sprintenddate"] = pd.to_datetime(spr["sprintenddate"], errors="coerce")
    spr.sort_values("sprintstartdate", inplace=True)
    spr["target_days"] = (spr["sprintenddate"] - spr["sprintstartdate"]).dt.days
    spr = spr[spr["target_days"] > 0].copy()
    
    iss_linked = iss.merge(summ[["issuekey", "sprintid"]], left_on="key", right_on="issuekey", how="inner")
    if "storypoints" in iss_linked.columns: iss_linked.rename(columns={"storypoints": "storypoint"}, inplace=True)
    iss_linked["prior_val"] = iss_linked["priority"].map(PRIORITY_MAP).fillna(1)
    
    agg_plan = iss_linked.groupby("sprintid").agg(
        planned_points=("storypoint", "sum"),
        planned_issues=("key", "count"),
        avg_priority=("prior_val", "mean")
    ).reset_index()
    
    done_mask = iss_linked["status"].astype(str).str.lower().isin(["done", "completed", "closed"])
    agg_done = iss_linked[done_mask].groupby("sprintid")["storypoint"].sum().reset_index().rename(columns={"storypoint": "completed_points"})
    
    df = spr.merge(agg_plan, on="sprintid", how="inner").merge(agg_done, on="sprintid", how="left").fillna(0)
    
    # Features
    df["velocity_lag_1"] = df["completed_points"].shift(1).fillna(df["completed_points"].mean())
    df["velocity_roll_3"] = df["completed_points"].rolling(3, min_periods=1).mean().shift(1).fillna(df["completed_points"].mean())
    df["days_roll_3"] = df["target_days"].rolling(3, min_periods=1).mean().shift(1).fillna(df["target_days"].mean())
    if "noofdevelopers" not in df.columns: df["noofdevelopers"] = 1
    df["points_per_dev"] = df["planned_points"] / df["noofdevelopers"].clip(lower=1)

    keep = ["planned_points", "planned_issues", "avg_priority", "noofdevelopers", "velocity_lag_1", "velocity_roll_3", "days_roll_3", "points_per_dev", "target_days"]
    return df[keep]

df = load_and_prep_enhanced()
X = df.drop("target_days", axis=1)
y = df["target_days"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [37]:
# -----------------------------
# 3️⃣ TUNING INNER MODELS
# -----------------------------
print("🔍 Pre-tuning base models for Ensemble...")

def get_tuned_est(model, params, name):
    # Standardize inside the search
    pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scl", RobustScaler()),
        ("est", model)
    ])
    grid = GridSearchCV(pipe, params, cv=5, scoring="r2")
    grid.fit(X_train, y_train)
    print(f"   Best {name} CV R2: {grid.best_score_:.3f}")
    return grid.best_estimator_

# 1. Random Forest
rf_best = get_tuned_est(
    RandomForestRegressor(random_state=42),
    {"est__n_estimators": [100], "est__max_depth": [5, 10], "est__min_samples_leaf": [1, 2]},
    "RandomForest"
)

# 2. XGBoost (Simpler)
xgb_best = get_tuned_est(
    XGBRegressor(random_state=42, objective="reg:squarederror"),  # standard mse often more stable than absolute
    {"est__n_estimators": [50, 100], "est__max_depth": [2, 3], "est__learning_rate": [0.05, 0.1], "est__reg_alpha": [0.1, 1.0]}, 
    "XGBoost"
)

# 3. SVR
svr_best = get_tuned_est(
    SVR(),
    {"est__C": [1, 10, 100], "est__kernel": ["rbf"]},
    "SVR"
)

# 4. Linear (No tuning needed usually)
linear = Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", RobustScaler()), ("est", LinearRegression())])

🔍 Pre-tuning base models for Ensemble...
   Best RandomForest CV R2: -0.305
   Best XGBoost CV R2: -0.035
   Best SVR CV R2: 0.324


In [38]:
# -----------------------------
# 4️⃣ ENSEMBLE & LOG-TRANSFORM
# -----------------------------
print("\n🏆 FINAL SHOWDOWN (V4.7 Ensemble + Log-Transform):")
print("-" * 50)

# Voting Regressor (Combine best 3 usually)
# Using RF (Non-linear), Linear (Trend), SVR (Smooth)
ensemble = VotingRegressor([
    ("rf", rf_best),
    ("lin", linear),
    ("svr", svr_best)
])

# Wrap w/ Log Transformation
models = {
    "Baseline (Mean)": DummyRegressor(strategy="mean"),
    "Linear Reg (Log)": TransformedTargetRegressor(regressor=linear, func=np.log1p, inverse_func=np.expm1),
    "RandomForest (Log)": TransformedTargetRegressor(regressor=rf_best, func=np.log1p, inverse_func=np.expm1),
    "XGBoost (Log)": TransformedTargetRegressor(regressor=xgb_best, func=np.log1p, inverse_func=np.expm1),
    "SVR (Log)": TransformedTargetRegressor(regressor=svr_best, func=np.log1p, inverse_func=np.expm1),
    "Voting (RF+Lin+SVR)": TransformedTargetRegressor(regressor=ensemble, func=np.log1p, inverse_func=np.expm1)
}

best_r2 = -float("inf")
best_model = None
best_name = ""

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    
    print(f"{name:<20}: MAE={mae:.2f} | R2={r2:.2f}")
    
    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_name = name

print(f"\n✅ WINNER: {best_name} with R2={best_r2:.2f}")
joblib.dump(best_model, os.path.join(MODEL_DIR, "deadline_model_v4_ensemble.joblib"))
print(f"💾 Model saved to {os.path.join(MODEL_DIR, 'deadline_model_v4_ensemble.joblib')}")


🏆 FINAL SHOWDOWN (V4.7 Ensemble + Log-Transform):
--------------------------------------------------
Baseline (Mean)     : MAE=3.17 | R2=-0.10
Linear Reg (Log)    : MAE=2.76 | R2=0.22
RandomForest (Log)  : MAE=2.15 | R2=0.49
XGBoost (Log)       : MAE=2.39 | R2=0.38
SVR (Log)           : MAE=1.86 | R2=0.66
Voting (RF+Lin+SVR) : MAE=2.16 | R2=0.52

✅ WINNER: SVR (Log) with R2=0.66
💾 Model saved to models\deadline_model_v4_ensemble.joblib
